In [225]:
# ============================================
# SETUP - Run this cell first
# ============================================
import pandas as pd
import matplotlib.pyplot as plt
import os

# SQLAlchemy imports
import sqlalchemy
from sqlalchemy import create_engine, inspect, func
from sqlalchemy import MetaData
import urllib.request

print("All libraries loaded!")
print(f"SQLAlchemy version: {sqlalchemy.__version__}")

All libraries loaded!
SQLAlchemy version: 2.0.46


In [226]:
# Download the Chinook sample database
chinook_url = 'https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite'

if not os.path.exists('chinook.db'):
    print('Downloading Chinook database...', end='')
    urllib.request.urlretrieve(chinook_url, 'chinook.db')
    print(' Done!')

assert os.path.exists('chinook.db'), "Database file not found!"
print(f"Database ready: chinook.db")

Database ready: chinook.db


# 🌟 Exercise 1 : Open the database
- open the database using sqlalchemy module interface. create an engine object in a variable named engine
- call the connect() method to obtain a connection and place in a variable named cur
now run the code below to to run reflecton on the database, prepare classes that map to the database and create an orm session :

```
### useful: extract classes from the chinook database
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

## we need to do this once
from sqlalchemy.ext.automap import automap_base

# produce a set of mappings from this MetaData.
Base = automap_base(metadata=metadata)

# calling prepare() just sets up mapped classes and relationships.
Base.prepare()

# also prepare an orm session
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()
```

### Solution

In [227]:
engine = create_engine('sqlite:///chinook.db')
connection = engine.connect()
print(f"Connected to: {engine.url}")

Connected to: sqlite:///chinook.db


In [228]:
### useful: extract classes from the chinook database
metadata = MetaData()
metadata.reflect(engine)

## we need to do this once
from sqlalchemy.ext.automap import automap_base

# produce a set of mappings from this MetaData.
Base = automap_base(metadata=metadata)

# calling prepare() just sets up mapped classes and relationships.
Base.prepare()

# also prepare an orm session
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()

print("Auto-generated classes:")
for class_name in sorted(Base.classes.keys()):
    print(f"  - {class_name}")

Auto-generated classes:
  - Album
  - Artist
  - Customer
  - Employee
  - Genre
  - Invoice
  - InvoiceLine
  - MediaType
  - Playlist
  - Track


In [229]:
# Helper functions
from IPython.display import display

def get_results(query):
    q = query.statement if hasattr(query, 'statement') else query
    return pd.read_sql(q, engine)

def display_results(query):
    df = get_results(query)
    display(df)
    return df

print("Helper functions ready!")

Helper functions ready!


# 🌟 Exercise 2 : table names
- print out all the table names

### Solution

In [230]:
inspector = inspect(engine)
table_names = inspector.get_table_names()

for i, table in enumerate(table_names):
  print(f" {i}. {table}")

 0. Album
 1. Artist
 2. Customer
 3. Employee
 4. Genre
 5. Invoice
 6. InvoiceLine
 7. MediaType
 8. Playlist
 9. PlaylistTrack
 10. Track


# 🌟 Exercise 3 : Tracks
- print out the first three tracks in the tracks table


### Solution

In [231]:
# Create class aliases
# Note: Chinook uses PascalCase table names (Artist, Album, Track, etc.)
Artist = Base.classes.Artist
Album = Base.classes.Album
Track = Base.classes.Track
Genre = Base.classes.Genre
Customer = Base.classes.Customer
Invoice = Base.classes.Invoice
InvoiceItem = Base.classes.InvoiceLine  # Note: Table is called "InvoiceLine" in Chinook
Employee = Base.classes.Employee  # For practice exercises

In [232]:
query = session.query(Track).limit(3)

# Loop through the results and print track.Name
for track in query:
  print(f"ID: {track.TrackId}: {track.Name}")


ID: 1: For Those About To Rock (We Salute You)
ID: 2: Balls to the Wall
ID: 3: Fast As a Shark


# 🌟 Exercise 4 : Albums from Tracks
- print out the track name and albums title of the first 20 tracks in the tracks table

### Solution

In [233]:
query = (
    session.query(
        Track.Name.label('track_name'),
        Album.Title.label('album_title')
    )
    .join(Album, Track.AlbumId == Album.AlbumId)
    .limit(20)
)

df = display_results(query)

,track_name,album_title
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Balls to the Wall,Balls to the Wall
2,Fast As a Shark,Restless and Wild
3,Restless and Wild,Restless and Wild
4,Princess of the Dawn,Restless and Wild
5,Put The Finger On You,For Those About To Rock We Salute You
6,Let's Get It Up,For Those About To Rock We Salute You
7,Inject The Venom,For Those About To Rock We Salute You
8,Snowballed,For Those About To Rock We Salute You
9,Evil Walks,For Those About To Rock We Salute You


# 🌟 Exercise : Tracks sold
- print out the first 10 track sales from the invoice_items table
- for these first 10 sales, print what are the names of the track sold, and the quantity sold

### Solution

In [234]:
# Print first 10 track sales
query = (
    session.query(
        InvoiceItem.InvoiceLineId,
        InvoiceItem.TrackId,
        InvoiceItem.Quantity,
        InvoiceItem.UnitPrice
    )
    .limit(10)
)

df = display_results(query)

,InvoiceLineId,TrackId,Quantity,UnitPrice
0,1,2,1,0.99
1,2,4,1,0.99
2,3,6,1,0.99
3,4,8,1,0.99
4,5,10,1,0.99
5,6,12,1,0.99
6,7,16,1,0.99
7,8,20,1,0.99
8,9,24,1,0.99
9,10,28,1,0.99


In [235]:
# Print first 10 tracks sold with names, quantity and price
query = (
    session.query(
        Track.Name.label('track_name'),
        InvoiceItem.Quantity.label('quantity'),
        InvoiceItem.UnitPrice.label('price')
    )
    .join(Track, InvoiceItem.TrackId == Track.TrackId)
    .limit(10)
)

df = display_results(query)

,track_name,quantity,price
0,Balls to the Wall,1,0.99
1,Restless and Wild,1,0.99
2,Put The Finger On You,1,0.99
3,Inject The Venom,1,0.99
4,Evil Walks,1,0.99
5,Breaking The Rules,1,0.99
6,Dog Eat Dog,1,0.99
7,Overdose,1,0.99
8,Love In An Elevator,1,0.99
9,Janie's Got A Gun,1,0.99


# 🌟 Exercise 6 : Top tracks sold
- print the names of top 10 tracks sold, and how many they times they were sold

### Solution

In [236]:
# Query:
# - Select Track.Name and func.sum(InvoiceItem.Quantity)
# - Join Track
# - Group by Track.TrackId
# - Order by sum descending
# - Limit 10

query = (
    session.query(
        Track.Name.label('track_name'),
        func.sum(InvoiceItem.Quantity).label('total_sold')
    )
    .join(Track, InvoiceItem.TrackId == Track.TrackId)
    .group_by(Track.TrackId)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
)

df_top_tracks = display_results(query)

,track_name,total_sold
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2
5,Not The Doctor,2
6,Por Causa De Você,2
7,Welcome Home (Sanitarium),2
8,Snowblind,2
9,Cornucopia,2


# 🌟 Exercise 7 : Top selling artists
- Who are the top 10 highest selling artists?

### Solution

In [237]:
# Join chain: InvoiceItem -> Track -> Album -> Artist
# Select: Artist.Name, sum of quantity, sum of (price * quantity)
# Store result in df_artists using: df_artists = display_results(query)

query = (
    session.query(
        Artist.Name.label('artist_name'),
        func.sum(InvoiceItem.Quantity).label('units_sold'),
        func.sum(InvoiceItem.UnitPrice * InvoiceItem.Quantity).label('revenue')
    )
    .join(Track, InvoiceItem.TrackId == Track.TrackId)
    .join(Album, Track.AlbumId == Album.AlbumId)
    .join(Artist, Album.ArtistId == Artist.ArtistId)
    .group_by(Artist.ArtistId)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
)

df_artists = display_results(query)


,artist_name,units_sold,revenue
0,Iron Maiden,140,138.60
1,U2,107,105.93
2,Metallica,91,90.09
3,Led Zeppelin,87,86.13
4,Os Paralamas Do Sucesso,45,44.55
5,Deep Purple,44,43.56
6,Faith No More,42,41.58
7,Lost,41,81.59
8,Eric Clapton,40,39.60
9,R.E.M.,39,38.61
